In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Arabic Handwritten Character Recognition
#  Adapted from the Kannada-MNIST version.
#
#  Transfer rationale
#  ──────────────────
#  Arabic is a cursive abjad script written right-to-left.  Characters change
#  shape depending on their position in a word (isolated, initial, medial,
#  final), and many letters are distinguished solely by dot diacritics above or
#  below the base form.  Key structural differences vs. Kannada:
#
#    • num_classes   : 10  → 66   (AHCD: 28 Arabic letters × isolated forms
#                                  + sub-letter variants as catalogued in the
#                                  dataset; exact count may vary by release –
#                                  a runtime warning fires on mismatch)
#    • image_size    : 28  → 32   (AHCD images are 32×32 greyscale)
#    • Dataset format: .npz arrays
#                      → folder-per-class PNG tree  (same layout as DHCD)
#    • Dataset loader: npz → Pillow file-walker (identical to Bengali version)
#    • Scaffold branch: 1×3 (Kannada short horizontals) → 3×3 + diagonal pair
#                       (3×3 captures connected cursive base strokes;
#                        a 3×3 dilated branch captures the dot diacritics
#                        which sit above/below the baseline at ~3-px offsets)
#    • STM kernels   : 1×3/3×1 → 3×3 + dilated  (cursive + dot context)
#    • Augmentation  : random_flip_left_right added – Arabic writers vary in
#                      pen pressure and slight lateral tilt, and the isolated-
#                      form dataset is already normalised for orientation, so
#                      mild horizontal flip generalises across writers without
#                      creating invalid glyphs (unlike vertical flip).
#    • Model name    : WhatNet-Kannada → WhatNet-Arabic
#    • Output files  : kannada_*.json  → arabic_*.json
#
#  Everything else – architecture depth, LR schedule, callbacks,
#  display utilities – is unchanged.
#
#  Dataset
#  ───────
#  AHCD – Arabic Handwritten Characters Dataset  (El-Sherif & Abdelazim, 2017)
#    • 16 800 images, 32×32 greyscale, folder-per-class layout
#    • 66 classes covering isolated forms of the 28 Arabic letters
#    • Kaggle:
#        https://www.kaggle.com/datasets/mloey1/ahcd1
#    • Expected directory structure after extraction:
#        data_dir/
#          Train/
#            1/  *.png   (class label 1 = ا  alef)
#            2/  *.png
#            …
#            66/ *.png
#          Test/
#            1/  *.png
#            …
#            66/ *.png
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Kannada version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 66 classes for AHCD (28 Arabic letters, isolated forms).
    #   Some Kaggle releases label 1-indexed folders 1–66.  The loader sorts
    #   folder names lexicographically and assigns 0-based indices, so the
    #   mapping is consistent regardless of the naming convention used.
    #   If your copy of the dataset has a different number of class folders,
    #   a runtime [WARN] is printed and you should update this value.
    "num_classes":      66,

    # ← CHANGED: AHCD images are 32×32 (matches original Bengali/Devanagari size)
    "image_size":       32,

    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # ← CHANGED: root of the AHCD folder-per-class tree.
    #   Kaggle notebook path (after adding the dataset):
    #     "/kaggle/input/ahcd1"
    #   Expected layout:  data_dir/Train/<class_folder>/*.png
    #                     data_dir/Test/<class_folder>/*.png
    "data_dir":    "/kaggle/input/datasets/mloey1/ahcd1",                   # ← CHANGED

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     ← CHANGED: AHCD uses a folder-per-class PNG tree (same layout as DHCD /
#     BanglaLekha), so we restore the Pillow file-walker from the Bengali
#     version.  The npz loader from the Kannada version is not needed.
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download AHCD from Kaggle:\n"
        "  https://www.kaggle.com/datasets/mloey1/ahcd1\n"
        "and extract so the path above contains Train/ and Test/ subdirs."
    )

def _collect_paths_and_labels(root: str):
    """
    Walk root/class_name/*.png (or .jpg / .bmp) and return parallel lists
    of file paths and 0-based integer labels sorted for reproducibility.
    Class index = sorted position of the folder name (lexicographic).
    """
    class_names = sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        folder = os.path.join(root, cls)
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                paths.append(os.path.join(folder, fname))
                labels.append(idx)
    return paths, labels, class_names

def _pil_load_fn(path_tensor, label_tensor):
    """
    tf.py_function wrapper: loads one image with Pillow, converts to
    greyscale, resizes to IMG×IMG, returns float32 (IMG, IMG, 1) in [0,255].
    """
    def _load(path_bytes):
        path = path_bytes.numpy().decode("utf-8")
        img  = PilImage.open(path).convert("L")
        img  = img.resize((IMG, IMG), PilImage.BILINEAR)
        arr  = np.array(img, dtype=np.float32)[:, :, np.newaxis]
        return arr
    img = tf.py_function(_load, [path_tensor], tf.float32)
    img.set_shape((IMG, IMG, 1))
    return img, label_tensor

def _make_ds_from_dir(root: str, shuffle: bool = False) -> tf.data.Dataset:
    paths, labels, _ = _collect_paths_and_labels(root)
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(_pil_load_fn, num_parallel_calls=AUTOTUNE)
    return ds

print("[INFO] Building Pillow-backed dataset (AHCD – Arabic) …")
train_full  = _make_ds_from_dir(os.path.join(CFG["data_dir"], "Train"), shuffle=True)
test_ds_raw = _make_ds_from_dir(os.path.join(CFG["data_dir"], "Test"),  shuffle=False)

# Sanity-check class count against CFG
_, _, _cls = _collect_paths_and_labels(os.path.join(CFG["data_dir"], "Train"))
print(f"[INFO] Classes found on disk: {len(_cls)}  |  CFG num_classes: {NUM_CLASSES}")
if len(_cls) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to {len(_cls)} to match your dataset.")

total        = sum(1 for _ in train_full)
n_val        = max(1, int(total * CFG["val_split"]))
n_train      = total - n_val
train_ds_raw = train_full.take(n_train)
val_ds_raw   = train_full.skip(n_train)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched from disk)")

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation – training only.

    Arabic isolated-form glyphs are naturally centred and upright in AHCD.
    Augmentations applied:
      • Brightness / contrast jitter  – simulates pen pressure variation
      • Pad-then-random-crop          – random sub-pixel translation
      • Random left-right flip        – Arabic writers vary slightly in lateral
                                        tilt; isolated glyphs remain valid
                                        under mild horizontal reflection.
    No vertical flip – it would invert dot positions above/below the baseline,
    creating invalid characters (e.g. ب ↔ ت confusion).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    img = tf.image.random_flip_left_right(img)   # ← CHANGED: added for Arabic
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Arabic stroke topology encoder.                           ← CHANGED docstring

    Arabic isolated-form glyphs are characterised by:
      • Sweeping connected cursive base strokes (horizontal + diagonal curves)
      • Dot diacritics sitting 2–4 px above or below the main body
      • A variety of ascender/descender shapes (ل ك ط etc.)

    Kernel design:
      • 3×3 standard branch   – captures the full cursive base stroke body
      • 3×3 dilated (rate=2)  – spans ~7 px; detects dots relative to the
                                 base form (dots are typically 2-3 px away)
      • 3×3 dilated (rate=3)  – wider context for letters with distant dots
                                 (e.g. ث  which has three dots well above)
      • 1×5 horizontal branch – retains sensitivity to the long horizontal
                                 baseline stroke present in ـ (tatweel) forms

    The 1×3 + 3×1 cross-hair from the Kannada version is replaced because
    Arabic dot diacritics require isotropic (square kernel) receptive fields
    to detect whether a dot is above, below, or beside the base glyph.
    """
    # ← CHANGED: 3×3 base + two dilated rates for dot-diacritic detection
    base = layers.Conv2D(64, 3, padding="same", use_bias=False, activation=gelu)(x)
    d2   = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d3   = layers.Conv2D(16, 3, padding="same", dilation_rate=3, use_bias=False, activation=gelu)(x)
    h    = layers.Conv2D(16, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([base, d2, d3, h])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed WhatNet-Arabic; num_classes=66; image_size=32.
#     Scaffold uses 3×3 (isotropic) to handle Arabic cursive strokes and dots.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_arabic(num_classes: int = 66, image_size: int = 32) -> Model:
    """
    WhatNet-Arabic: WhatNet adapted for AHCD Arabic character recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (overall glyph body)
      • Dot-aware scaffold: 3×3 conv – isotropic kernel captures both the
        cursive base stroke AND nearby dot diacritics in a single receptive
        field; combined with dilated branches in STM for far-dot detection
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 3×3 + dilated (rate 2 & 3) + 1×5
      • Gated fusion → final MLP + layer norm → logits (66 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×3 (Kannada) → 3×3 (Arabic isotropic dot/curve)
    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Arabic")       # ← CHANGED

MODELS_TF = {
    "WhatNet-Arabic": lambda: build_whatnet_arabic(NUM_CLASSES, IMG),  # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [AHCD – Arabic]", "bold"))                                # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "arabic_results.json")    # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "arabic_histories.json") # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] AHCD Arabic benchmark complete.\n", "green", "bold"))   # ← CHANGED

2026-04-20 10:28:57.275489: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776680937.436819      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776680937.483756      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776680937.855473      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776680937.855509      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776680937.855512      55 computation_placer.cc:177] computation placer alr

[INFO] Building Pillow-backed dataset (AHCD – Arabic) …


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/mloey1/ahcd1/Train'

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Arabic Handwritten Character Recognition
#  Adapted from the Kannada-MNIST version.
#
#  Transfer rationale
#  ──────────────────
#  Arabic is a cursive abjad script written right-to-left.  Characters change
#  shape depending on their position in a word (isolated, initial, medial,
#  final), and many letters are distinguished solely by dot diacritics above or
#  below the base form.  Key structural differences vs. Kannada:
#
#    • num_classes   : 10  → 66   (AHCD: 28 Arabic letters × isolated forms
#                                  + sub-letter variants as catalogued in the
#                                  dataset; exact count may vary by release –
#                                  a runtime warning fires on mismatch)
#    • image_size    : 28  → 32   (AHCD images are 32×32 greyscale)
#    • Dataset format: .npz arrays
#                      → folder-per-class PNG tree  (same layout as DHCD)
#    • Dataset loader: npz → Pillow file-walker (identical to Bengali version)
#    • Scaffold branch: 1×3 (Kannada short horizontals) → 3×3 + diagonal pair
#                       (3×3 captures connected cursive base strokes;
#                        a 3×3 dilated branch captures the dot diacritics
#                        which sit above/below the baseline at ~3-px offsets)
#    • STM kernels   : 1×3/3×1 → 3×3 + dilated  (cursive + dot context)
#    • Augmentation  : random_flip_left_right added – Arabic writers vary in
#                      pen pressure and slight lateral tilt, and the isolated-
#                      form dataset is already normalised for orientation, so
#                      mild horizontal flip generalises across writers without
#                      creating invalid glyphs (unlike vertical flip).
#    • Model name    : WhatNet-Kannada → WhatNet-Arabic
#    • Output files  : kannada_*.json  → arabic_*.json
#
#  Everything else – architecture depth, LR schedule, callbacks,
#  display utilities – is unchanged.
#
#  Dataset
#  ───────
#  AHCD – Arabic Handwritten Characters Dataset  (El-Sherif & Abdelazim, 2017)
#    • 16 800 images, 32×32 greyscale, folder-per-class layout
#    • 66 classes covering isolated forms of the 28 Arabic letters
#    • Kaggle:
#        https://www.kaggle.com/datasets/mloey1/ahcd1
#    • Expected directory structure after extraction:
#        data_dir/
#          Train/
#            1/  *.png   (class label 1 = ا  alef)
#            2/  *.png
#            …
#            66/ *.png
#          Test/
#            1/  *.png
#            …
#            66/ *.png
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE  — CSV loader replaces the Pillow file-walker
# ─────────────────────────────────────────────────────────────────────────────
import glob, os
import numpy as np
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE
IMG      = CFG["image_size"]   # 32
BS       = CFG["batch_size"]
SEED     = 42


def _find_csv(data_dir: str, pattern: str) -> str:
    """
    Locate a CSV file whose name contains `pattern` (case-insensitive).
    Raises a clear FileNotFoundError if nothing matches.
    """
    candidates = [
        p for p in glob.glob(os.path.join(data_dir, "*.csv"))
        if pattern.lower() in os.path.basename(p).lower()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"[ERROR] No CSV matching '*{pattern}*' found in {data_dir}\n"
            f"  Files present: {os.listdir(data_dir)}"
        )
    # Prefer the shortest name (most direct match)
    return sorted(candidates, key=len)[0]


def _load_csv_dataset(data_dir: str, split: str):
    """
    Load one split ('Train' or 'Test') from the flat CSV files.

    Returns
    -------
    images : float32 ndarray, shape (N, 32, 32, 1), values in [0, 255]
    labels : int32 ndarray,   shape (N,),            values in [0, 27]
    """
    img_path = _find_csv(data_dir, f"csv{split}Images")
    lbl_path = _find_csv(data_dir, f"csv{split}Label")

    print(f"[INFO] Loading {split} images from : {os.path.basename(img_path)}")
    print(f"[INFO] Loading {split} labels from : {os.path.basename(lbl_path)}")

    # Read raw arrays — no header row in either file
    images_flat = np.loadtxt(img_path, delimiter=",", dtype=np.float32)  # (N, 1024)
    labels_raw  = np.loadtxt(lbl_path, delimiter=",", dtype=np.int32)    # (N,) or (N, 1)

    labels_raw = labels_raw.ravel()           # ensure 1-D
    labels     = labels_raw - 1              # shift 1-based → 0-based  [0, 27]

    # Reshape: (N, 1024) → (N, 32, 32, 1)
    N = images_flat.shape[0]
    images = images_flat.reshape(N, IMG, IMG, 1)

    # Sanity checks
    n_classes = int(labels.max()) + 1
    print(f"[INFO] {split}: {N:,} samples, {n_classes} classes detected "
          f"(label range {labels.min()}–{labels.max()})")
    if n_classes != CFG["num_classes"]:
        print(f"[WARN] Label range implies {n_classes} classes but "
              f"CFG['num_classes']={CFG['num_classes']} — update CFG to match.")

    return images, labels


# ── Load raw arrays ───────────────────────────────────────────────────────────
print("[INFO] Building CSV-backed dataset (AHCD – Arabic) …")
train_images, train_labels = _load_csv_dataset(CFG["data_dir"], "Train")
test_images,  test_labels  = _load_csv_dataset(CFG["data_dir"], "Test")

# ── Train / val split ─────────────────────────────────────────────────────────
N_train_full = train_images.shape[0]
n_val        = max(1, int(N_train_full * CFG["val_split"]))
n_train      = N_train_full - n_val

# Shuffle before splitting so val isn't just the last N samples
rng   = np.random.default_rng(SEED)
perm  = rng.permutation(N_train_full)
train_images = train_images[perm]
train_labels = train_labels[perm]

tr_imgs, tr_lbls = train_images[:n_train], train_labels[:n_train]
vl_imgs, vl_lbls = train_images[n_train:], train_labels[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {test_images.shape[0]:,}")

NUM_CLASSES = CFG["num_classes"]


# ── Preprocessing helpers (unchanged from original) ───────────────────────────
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    img = tf.image.random_flip_left_right(img)
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)


# ── Build tf.data pipelines ───────────────────────────────────────────────────
def _make_ds(images, labels, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(images), tf.constant(labels, dtype=tf.int32))
    )
    ds = ds.map(normalise, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(8192, seed=SEED)
    ds = ds.map(to_onehot, num_parallel_calls=AUTOTUNE)
    return ds.batch(BS).prefetch(AUTOTUNE)

train_ds = _make_ds(tr_imgs, tr_lbls, training=True)
val_ds   = _make_ds(vl_imgs, vl_lbls, training=False)

# Test pipeline without one-hot (used by compute_macro_f1)
test_ds = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(test_images), tf.constant(test_labels, dtype=tf.int32))
    )
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

# Test pipeline with one-hot (used by model.evaluate)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(test_images), tf.constant(test_labels, dtype=tf.int32))
    )
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Arabic stroke topology encoder.                           ← CHANGED docstring

    Arabic isolated-form glyphs are characterised by:
      • Sweeping connected cursive base strokes (horizontal + diagonal curves)
      • Dot diacritics sitting 2–4 px above or below the main body
      • A variety of ascender/descender shapes (ل ك ط etc.)

    Kernel design:
      • 3×3 standard branch   – captures the full cursive base stroke body
      • 3×3 dilated (rate=2)  – spans ~7 px; detects dots relative to the
                                 base form (dots are typically 2-3 px away)
      • 3×3 dilated (rate=3)  – wider context for letters with distant dots
                                 (e.g. ث  which has three dots well above)
      • 1×5 horizontal branch – retains sensitivity to the long horizontal
                                 baseline stroke present in ـ (tatweel) forms

    The 1×3 + 3×1 cross-hair from the Kannada version is replaced because
    Arabic dot diacritics require isotropic (square kernel) receptive fields
    to detect whether a dot is above, below, or beside the base glyph.
    """
    # ← CHANGED: 3×3 base + two dilated rates for dot-diacritic detection
    base = layers.Conv2D(64, 3, padding="same", use_bias=False, activation=gelu)(x)
    d2   = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d3   = layers.Conv2D(16, 3, padding="same", dilation_rate=3, use_bias=False, activation=gelu)(x)
    h    = layers.Conv2D(16, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([base, d2, d3, h])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed WhatNet-Arabic; num_classes=66; image_size=32.
#     Scaffold uses 3×3 (isotropic) to handle Arabic cursive strokes and dots.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_arabic(num_classes: int = 66, image_size: int = 32) -> Model:
    """
    WhatNet-Arabic: WhatNet adapted for AHCD Arabic character recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (overall glyph body)
      • Dot-aware scaffold: 3×3 conv – isotropic kernel captures both the
        cursive base stroke AND nearby dot diacritics in a single receptive
        field; combined with dilated branches in STM for far-dot detection
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 3×3 + dilated (rate 2 & 3) + 1×5
      • Gated fusion → final MLP + layer norm → logits (66 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×3 (Kannada) → 3×3 (Arabic isotropic dot/curve)
    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Arabic")       # ← CHANGED

MODELS_TF = {
    "WhatNet-Arabic": lambda: build_whatnet_arabic(NUM_CLASSES, IMG),  # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [AHCD – Arabic]", "bold"))                                # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "arabic_results.json")    # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "arabic_histories.json") # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] AHCD Arabic benchmark complete.\n", "green", "bold"))   # ← CHANGED

[INFO] Building CSV-backed dataset (AHCD – Arabic) …
[INFO] Loading Train images from : csvTrainImages 13440x1024.csv
[INFO] Loading Train labels from : csvTrainLabel 13440x1.csv
[INFO] Train: 13,440 samples, 28 classes detected (label range 0–27)
[WARN] Label range implies 28 classes but CFG['num_classes']=66 — update CFG to match.
[INFO] Loading Test images from : csvTestImages 3360x1024.csv
[INFO] Loading Test labels from : csvTestLabel 3360x1.csv
[INFO] Test: 3,360 samples, 28 classes detected (label range 0–27)
[WARN] Label range implies 28 classes but CFG['num_classes']=66 — update CFG to match.
[INFO] Train: 12,096 | Val: 1,344 | Test: 3,360


I0000 00:00:1776681202.523589      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776681202.529314      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [AHCD – Arabic]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Arabic  –  Parameter Summary                        ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║              288  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_normal

I0000 00:00:1776681229.954364     140 service.cc:152] XLA service 0x7ebdc801be20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776681229.954446     140 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776681229.954453     140 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776681234.538481     140 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776681257.338619     140 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/50  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=1.9264  acc=46.90%  val_loss=5.3864  val_acc=2.98%  lr=5.00e-04  [73.4s]
  Epoch   2/50  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.8596  acc=84.37%  val_loss=4.3040  val_acc=7.29%  lr=4.98e-04  [17.5s]
  Epoch   3/50  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.6771  acc=91.45%  val_loss=1.1674  val_acc=74.70%  lr=4.96e-04  [17.9s]
  Epoch   4/50  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.6080  acc=94.15%  val_loss=0.6952  val_acc=90.25%  lr=4.92e-04  [18.3s]
  Epoch   5/50  ██░░░░░░░░░░░░░░░░░░  10.0%  loss=0.5723  acc=95.10%  val_loss=0.9369  val_acc=81.25%  lr=4.88e-04  [17.9s]
  Epoch   6/50  ██░░░░░░░░░░░░░░░░░░  12.0%  loss=0.5555  acc=95.71%  val_loss=0.7007  val_acc=90.33%  lr=4.82e-04  [19.2s]
  Epoch   7/50  ██░░░░░░░░░░░░░░░░░░  14.0%  loss=0.5252  acc=96.53%  val_loss=0.6765  val_acc=92.04%  lr=4.76e-04  [20.3s]
  Epoch   8/50  ███░░░░░░░░░░░░░░░░░  16.0%  loss=0.5141  acc=96.78%  val_loss=0.6060  val_acc=94.72%  lr=4.69e-04  [20.9s]
  Epoch   

# Method

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Arabic Handwritten Character Recognition
#  Adapted from the Kannada-MNIST version.
#
#  Transfer rationale
#  ──────────────────
#  Arabic is a cursive abjad script written right-to-left.  Characters change
#  shape depending on their position in a word (isolated, initial, medial,
#  final), and many letters are distinguished solely by dot diacritics above or
#  below the base form.  Key structural differences vs. Kannada:
#
#    • num_classes   : 10  → 66   (AHCD: 28 Arabic letters × isolated forms
#                                  + sub-letter variants as catalogued in the
#                                  dataset; exact count may vary by release –
#                                  a runtime warning fires on mismatch)
#    • image_size    : 28  → 32   (AHCD images are 32×32 greyscale)
#    • Dataset format: .npz arrays
#                      → folder-per-class PNG tree  (same layout as DHCD)
#    • Dataset loader: npz → Pillow file-walker (identical to Bengali version)
#    • Scaffold branch: 1×3 (Kannada short horizontals) → 3×3 + diagonal pair
#                       (3×3 captures connected cursive base strokes;
#                        a 3×3 dilated branch captures the dot diacritics
#                        which sit above/below the baseline at ~3-px offsets)
#    • STM kernels   : 1×3/3×1 → 3×3 + dilated  (cursive + dot context)
#    • Augmentation  : random_flip_left_right added – Arabic writers vary in
#                      pen pressure and slight lateral tilt, and the isolated-
#                      form dataset is already normalised for orientation, so
#                      mild horizontal flip generalises across writers without
#                      creating invalid glyphs (unlike vertical flip).
#    • Model name    : WhatNet-Kannada → WhatNet-Arabic
#    • Output files  : kannada_*.json  → arabic_*.json
#
#  Everything else – architecture depth, LR schedule, callbacks,
#  display utilities – is unchanged.
#
#  Dataset
#  ───────
#  AHCD – Arabic Handwritten Characters Dataset  (El-Sherif & Abdelazim, 2017)
#    • 16 800 images, 32×32 greyscale, folder-per-class layout
#    • 66 classes covering isolated forms of the 28 Arabic letters
#    • Kaggle:
#        https://www.kaggle.com/datasets/mloey1/ahcd1
#    • Expected directory structure after extraction:
#        data_dir/
#          Train/
#            1/  *.png   (class label 1 = ا  alef)
#            2/  *.png
#            …
#            66/ *.png
#          Test/
#            1/  *.png
#            …
#            66/ *.png
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE  — CSV loader replaces the Pillow file-walker
# ─────────────────────────────────────────────────────────────────────────────
import glob, os
import numpy as np
import tensorflow as tf

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Fields marked  ← CHANGED  differ from the Kannada version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 66 classes for AHCD (28 Arabic letters, isolated forms).
    #   Some Kaggle releases label 1-indexed folders 1–66.  The loader sorts
    #   folder names lexicographically and assigns 0-based indices, so the
    #   mapping is consistent regardless of the naming convention used.
    #   If your copy of the dataset has a different number of class folders,
    #   a runtime [WARN] is printed and you should update this value.
    "num_classes":      28,

    # ← CHANGED: AHCD images are 32×32 (matches original Bengali/Devanagari size)
    "image_size":       32,

    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # ← CHANGED: root of the AHCD folder-per-class tree.
    #   Kaggle notebook path (after adding the dataset):
    #     "/kaggle/input/ahcd1"
    #   Expected layout:  data_dir/Train/<class_folder>/*.png
    #                     data_dir/Test/<class_folder>/*.png
    "data_dir":    "/kaggle/input/datasets/mloey1/ahcd1/",                   # ← CHANGED

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE


def _find_csv(data_dir: str, pattern: str) -> str:
    """
    Locate a CSV file whose name contains `pattern` (case-insensitive).
    Raises a clear FileNotFoundError if nothing matches.
    """
    candidates = [
        p for p in glob.glob(os.path.join(data_dir, "*.csv"))
        if pattern.lower() in os.path.basename(p).lower()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"[ERROR] No CSV matching '*{pattern}*' found in {data_dir}\n"
            f"  Files present: {os.listdir(data_dir)}"
        )
    # Prefer the shortest name (most direct match)
    return sorted(candidates, key=len)[0]


def _load_csv_dataset(data_dir: str, split: str):
    """
    Load one split ('Train' or 'Test') from the flat CSV files.

    Returns
    -------
    images : float32 ndarray, shape (N, 32, 32, 1), values in [0, 255]
    labels : int32 ndarray,   shape (N,),            values in [0, 27]
    """
    img_path = _find_csv(data_dir, f"csv{split}Images")
    lbl_path = _find_csv(data_dir, f"csv{split}Label")

    print(f"[INFO] Loading {split} images from : {os.path.basename(img_path)}")
    print(f"[INFO] Loading {split} labels from : {os.path.basename(lbl_path)}")

    # Read raw arrays — no header row in either file
    images_flat = np.loadtxt(img_path, delimiter=",", dtype=np.float32)  # (N, 1024)
    labels_raw  = np.loadtxt(lbl_path, delimiter=",", dtype=np.int32)    # (N,) or (N, 1)

    labels_raw = labels_raw.ravel()           # ensure 1-D
    labels     = labels_raw - 1              # shift 1-based → 0-based  [0, 27]

    # Reshape: (N, 1024) → (N, 32, 32, 1)
    N = images_flat.shape[0]
    images = images_flat.reshape(N, IMG, IMG, 1)

    # Sanity checks
    n_classes = int(labels.max()) + 1
    print(f"[INFO] {split}: {N:,} samples, {n_classes} classes detected "
          f"(label range {labels.min()}–{labels.max()})")
    if n_classes != CFG["num_classes"]:
        print(f"[WARN] Label range implies {n_classes} classes but "
              f"CFG['num_classes']={CFG['num_classes']} — update CFG to match.")

    return images, labels


# ── Load raw arrays ───────────────────────────────────────────────────────────
print("[INFO] Building CSV-backed dataset (AHCD – Arabic) …")
train_images, train_labels = _load_csv_dataset(CFG["data_dir"], "Train")
test_images,  test_labels  = _load_csv_dataset(CFG["data_dir"], "Test")

# ── Train / val split ─────────────────────────────────────────────────────────
N_train_full = train_images.shape[0]
n_val        = max(1, int(N_train_full * CFG["val_split"]))
n_train      = N_train_full - n_val

# Shuffle before splitting so val isn't just the last N samples
rng   = np.random.default_rng(SEED)
perm  = rng.permutation(N_train_full)
train_images = train_images[perm]
train_labels = train_labels[perm]

tr_imgs, tr_lbls = train_images[:n_train], train_labels[:n_train]
vl_imgs, vl_lbls = train_images[n_train:], train_labels[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {test_images.shape[0]:,}")

NUM_CLASSES = CFG["num_classes"]


# ── Preprocessing helpers (unchanged from original) ───────────────────────────
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    img = tf.image.random_flip_left_right(img)
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)


# ── Build tf.data pipelines ───────────────────────────────────────────────────
def _make_ds(images, labels, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(images), tf.constant(labels, dtype=tf.int32))
    )
    ds = ds.map(normalise, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(8192, seed=SEED)
    ds = ds.map(to_onehot, num_parallel_calls=AUTOTUNE)
    return ds.batch(BS).prefetch(AUTOTUNE)

train_ds = _make_ds(tr_imgs, tr_lbls, training=True)
val_ds   = _make_ds(vl_imgs, vl_lbls, training=False)

# Test pipeline without one-hot (used by compute_macro_f1)
test_ds = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(test_images), tf.constant(test_labels, dtype=tf.int32))
    )
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

# Test pipeline with one-hot (used by model.evaluate)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(test_images), tf.constant(test_labels, dtype=tf.int32))
    )
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Arabic stroke topology encoder.                           ← CHANGED docstring

    Arabic isolated-form glyphs are characterised by:
      • Sweeping connected cursive base strokes (horizontal + diagonal curves)
      • Dot diacritics sitting 2–4 px above or below the main body
      • A variety of ascender/descender shapes (ل ك ط etc.)

    Kernel design:
      • 3×3 standard branch   – captures the full cursive base stroke body
      • 3×3 dilated (rate=2)  – spans ~7 px; detects dots relative to the
                                 base form (dots are typically 2-3 px away)
      • 3×3 dilated (rate=3)  – wider context for letters with distant dots
                                 (e.g. ث  which has three dots well above)
      • 1×5 horizontal branch – retains sensitivity to the long horizontal
                                 baseline stroke present in ـ (tatweel) forms

    The 1×3 + 3×1 cross-hair from the Kannada version is replaced because
    Arabic dot diacritics require isotropic (square kernel) receptive fields
    to detect whether a dot is above, below, or beside the base glyph.
    """
    # ← CHANGED: 3×3 base + two dilated rates for dot-diacritic detection
    base = layers.Conv2D(64, 3, padding="same", use_bias=False, activation=gelu)(x)
    d2   = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d3   = layers.Conv2D(16, 3, padding="same", dilation_rate=3, use_bias=False, activation=gelu)(x)
    h    = layers.Conv2D(16, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([base, d2, d3, h])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     ← CHANGED: renamed WhatNet-Arabic; num_classes=66; image_size=32.
#     Scaffold uses 3×3 (isotropic) to handle Arabic cursive strokes and dots.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_arabic(num_classes: int = 66, 
    drop_path_rate: float = 0.05,
    dropout_rate: float = 0.3,
    weight_decay: float = 1e-4,
    head_units: int = 256,
    override_tier: int = None,
    image_size: int = 32) -> Model:
    """
    WhatNet-Arabic: WhatNet adapted for AHCD Arabic character recognition.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (overall glyph body)
      • Dot-aware scaffold: 3×3 conv – isotropic kernel captures both the
        cursive base stroke AND nearby dot diacritics in a single receptive
        field; combined with dilated branches in STM for far-dot detection
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – 3×3 + dilated (rate 2 & 3) + 1×5
      • Gated fusion → final MLP + layer norm → logits (66 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    # ← CHANGED scaffold: 1×3 (Kannada) → 3×3 (Arabic isotropic dot/curve)
    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # # ── Decoder head ──────────────────────────────────────────────────────
    # cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    # afc_in   = layers.GlobalAveragePooling2D()(enc3)
    # afc_in   = layers.Add()([afc_in, cstb_out])
    # afc_out  = adaptive_filter_capsule(afc_in, K)

    # stgm_out    = stroke_topology_module(enc3, 256)
    # combined    = layers.Concatenate()([stgm_out, afc_out])
    # gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    # stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    # fused       = layers.Concatenate()([stgm_scaled, afc_out])

    # x   = layers.Dense(512)(fused)
    # x   = layers.LayerNormalization()(x)
    # x   = layers.Activation(gelu)(x)
    # out = layers.Dense(K, name="logits")(x)

# ── Multi-scale GAP fusion ──────────────────────────────────────────────
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    # ── Adaptive Filter Capsule (AFC) ───────────────────────────────────────
    # Projects the fused multi-scale vector into capsule space.
    # Each of the K capsules learns to respond to one character class.
    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # ── Classification head ─────────────────────────────────────────────────
    # Dense projection of the raw GAP features (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # ── Gated fusion: AFC scores + dense-head logits ────────────────────────
    # A learned scalar gate (per-sample softmax over 2 weights) blends the
    # AFC capsule scores with the plain dense logits.  This lets the model
    # learn how much to trust the capsule routing vs. the direct projection.
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)

    # gate[:,0] weights the dense head; gate[:,1] weights the AFC output
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out,gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=outputs, name="WhatNet-Arabic")       # ← CHANGED

MODELS_TF = {
    "WhatNet-Arabic": lambda: build_whatnet_arabic(NUM_CLASSES, IMG),  # ← CHANGED
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [AHCD – Arabic]", "bold"))                                # ← CHANGED
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
        # EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "arabic_results.json")    # ← CHANGED
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "arabic_histories.json") # ← CHANGED
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] AHCD Arabic benchmark complete.\n", "green", "bold"))   # ← CHANGED

[INFO] Building CSV-backed dataset (AHCD – Arabic) …
[INFO] Loading Train images from : csvTrainImages 13440x1024.csv
[INFO] Loading Train labels from : csvTrainLabel 13440x1.csv
[INFO] Train: 13,440 samples, 28 classes detected (label range 0–27)
[INFO] Loading Test images from : csvTestImages 3360x1024.csv
[INFO] Loading Test labels from : csvTestLabel 3360x1.csv
[INFO] Test: 3,360 samples, 28 classes detected (label range 0–27)
[INFO] Train: 12,096 | Val: 1,344 | Test: 3,360

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [AHCD – Arabic]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Arabic  –  Parameter Summary                        ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════

In [4]:
n_actual = len(np.unique(train_labels))
print(n_actual)  # probably 28, not 66

28


In [ ]:
✔ Done: best val acc = 98.44%  wall time = 1588s

╔══════════════════════════════════════════════════════════════════════╗
║  FINAL TEST-SET COMPARISON                                           ║
╠════════════════════════╦════════════╦════════════╦════════════╦══════╣
║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║
╠════════════════════════╬════════════╬════════════╬════════════╬══════╣
║★ WhatNet-Arabic        ║ 5,423,110 ║     98.10%║     98.09%║0.701 ║
╚════════════════════════╩════════════╩════════════╩════════════╩══════╝

  ★  Winner: WhatNet-Arabic  (98.10% test accuracy)